# TTPTrace — İlk Oturum Analizi

**Deney:** `TTPT-20260929-B001` · **Ortam:** Windows / Sysmon · **Oturum:** benign

Bu defter, kontrollü laboratuvar oturumunun JSONL olaylarını doğrular; süreç, ağ ve DNS kayıtlarını ilişkilendirir ve süreç bazında model özellikleri üretir.

## Bu aşamanın hedefi

- Manifest, zaman aralığı, kimlik ve olay sayısı kontrollerini yapmak.
- Oturum içi ve oturum öncesi süreç bağlamını `ProcessGuid` ile birleştirmek.
- Ağ, DNS ve ebeveyn-çocuk ilişkilerinden süreç özellikleri çıkarmak.
- Eksik bağlamın etkisini ölçmek ve doğrulanmış JSONL çıktısı üretmek.
- Sayısal model özellikleriyle inceleme metaverilerini ayrı pandas tablolarına dönüştürmek.

**Beklenen:** 269 olay, 194 ağ olayı (ID 3), 15 DNS olayı (ID 22), 23 süreç oluşturma olayı (ID 1) ve dokuz oturum öncesi bağlam kaydı. Genel özet 33 süreci kapsar.

Bu sayılar veri hazırlama kontrolleridir; model eğitimi veya model başarımı değildir. ATT&CK etiketleme ve çoklu oturum modeli sonraki aşamalarda geliştirilecektir.

## Çalıştırma düzeni

Hücreleri yukarıdan aşağıya çalıştırın. Colab oturumu yenilendiğinde `events.jsonl`, `analysis-summary.json`, `manifest.json`, `process-context.jsonl` ve `process-context-additional.jsonl` dosyalarını `/content` altına yeniden yükleyin. Notebook Drive'da saklanır; `/content` altındaki veriler geçicidir.

Başlangıç için Colab CPU ortamı ve hazır gelen pandas paketi yeterlidir.


In [ ]:
# 1. Dosya konumları ve yeniden yükleme
import json
import hashlib
import sys
from pathlib import Path
from collections import Counter

DATA_DIR = Path("/content")
REQUIRED_FILES = ["events.jsonl", "analysis-summary.json", "manifest.json"]

missing = [name for name in REQUIRED_FILES if not (DATA_DIR / name).is_file()]
if missing:
    from google.colab import files
    print("Eksik dosyaları seç:", ", ".join(missing))
    files.upload()

missing = [name for name in REQUIRED_FILES if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Eksik dosyalar: {missing}")

print("Python:", sys.version.split()[0])
for name in REQUIRED_FILES:
    print(f"Hazır: {name} ({(DATA_DIR / name).stat().st_size} bayt)")

In [ ]:
# 2. Aktarım bütünlüğü ve oturum bilgisi
# Bunlar bu oturumun yerel dosyalarından hesaplanan SHA-256 değerleridir.
EXPECTED_SHA256 = {
    "events.jsonl": "040a2a1174aa2c169ace386d253000ab8e73fb14cb135fd7839153abcadaf8e4",
    "analysis-summary.json": "62a42d1b8e476ce6424f33d6f0425b646f45d229b35dbd33a8d519d33200efad",
    "manifest.json": "b2b256cfc37a0a8f44afdf250b3cc4a6f62b7fbafc5b81e526e6fb25cf7edd91",
}

for name, expected in EXPECTED_SHA256.items():
    actual = hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Dosya bütünlüğü uyuşmuyor: {name}")
    print(f"SHA-256 doğrulandı: {name}")

manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
summary = json.loads((DATA_DIR / "analysis-summary.json").read_text(encoding="utf-8"))
if manifest["experiment_id"] != summary["experiment_id"]:
    raise ValueError("Manifest ile analiz özeti farklı oturumlara ait.")

print("Deney:", manifest["experiment_id"])
print("Oturum türü:", manifest["session_type"])

In [ ]:
# 3. JSONL dosyasını olay listesine dönüştürme
def load_events(path: Path) -> list[dict]:
    events = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                event = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Satır {line_number}: geçersiz JSON") from exc
            if not isinstance(event, dict):
                raise ValueError(f"Satır {line_number}: JSON nesnesi bekleniyordu")
            events.append(event)
    return events

events = load_events(DATA_DIR / "events.jsonl")
print("Olay sayısı:", len(events))
print("İlk olayın alanları:", ", ".join(events[0].keys()) if events else "Dosya boş")

In [ ]:
# 4. Önceki analizle sayısal karşılaştırma
counts = Counter(event["event_id"] for event in events)
expected_counts = {1: 23, 3: 194, 5: 28, 11: 3, 12: 3, 13: 3, 22: 15}

checks = {
    "Toplam olay": len(events) == summary["analyzed_event_count"] == 269,
    "Ağ olayları": counts[3] == summary["network_event_count"] == 194,
    "DNS olayları": counts[22] == summary["dns_event_count"] == 15,
    "Olay türü dağılımı": dict(counts) == expected_counts,
    "Oturum kimlikleri": all(
        event["experiment_id"] == manifest["experiment_id"]
        and event["host_id"] == manifest["host"]["host_id"]
        for event in events
    ),
}

for event_id, count in sorted(counts.items()):
    print(f"Event ID {event_id:>2}: {count}")
print()
for name, passed in checks.items():
    print(f"{name}: {'OK' if passed else 'UYUŞMUYOR'}")

if not all(checks.values()):
    raise ValueError("Kontrollerden biri uyuşmuyor; çıktıyı incele.")

print("\nColab aktarımı ve başlangıç kontrolleri tamamlandı.")

In [ ]:
def filter_events(events: list[dict], event_id: int) -> list[dict]:
    selected = []

    for event in events:
        if event["event_id"] == event_id:
            selected.append(event)

    return selected


process_events = filter_events(events, 1)
network_events = filter_events(events, 3)
dns_events = filter_events(events, 22)

print("Süreç oluşturma:", len(process_events))
print("Ağ olayları:", len(network_events))
print("DNS olayları:", len(dns_events))

In [ ]:
def build_process_index(process_events: list[dict]) -> dict[str, dict]:
    process_index = {}

    for event in process_events:
        process_guid = event["data"]["ProcessGuid"]

        if process_guid in process_index:
            raise ValueError(f"Tekrarlanan ProcessGuid: {process_guid}")

        process_index[process_guid] = event

    return process_index


process_index = build_process_index(process_events)

whoami_guid = "{14bdcb74-dd88-6aba-1c01-000000000600}"
whoami_event = process_index[whoami_guid]

parent_guid = whoami_event["data"]["ParentProcessGuid"]
parent_event = process_index.get(parent_guid)

print("İndeksteki süreç sayısı:", len(process_index))
print("Çocuk süreç:", whoami_event["data"]["Image"])
print("Ebeveyn GUID:", parent_guid)

if parent_event is None:
    print("Ebeveynin oluşturma kaydı bu oturumun içinde bulunamadı.")
else:
    print("Ebeveyn süreç:", parent_event["data"]["Image"])
    print("Ebeveyn komutu:", parent_event["data"]["CommandLine"])

In [ ]:
def link_events_to_processes(
    events: list[dict],
    process_index: dict[str, dict],
) -> list[dict]:
    linked_events = []

    for event in events:
        process_guid = event["data"]["ProcessGuid"]
        process_event = process_index.get(process_guid)

        linked_events.append({
            "record_id": event["record_id"],
            "utc_time": event["utc_time"],
            "process_guid": process_guid,
            "image": event["data"]["Image"],
            "process_creation_found": process_event is not None,
            "command_line": (
                process_event["data"]["CommandLine"]
                if process_event is not None
                else None
            ),
        })

    return linked_events


network_links = link_events_to_processes(network_events, process_index)

matched_count = sum(
    link["process_creation_found"] for link in network_links
)
unmatched_count = len(network_links) - matched_count

print("Toplam ağ olayı:", len(network_links))
print("Süreç oluşturma kaydı bulunan:", matched_count)
print("Süreç oluşturma kaydı bulunamayan:", unmatched_count)
print("Sayı kontrolü:", matched_count + unmatched_count == len(network_events))

matched_example = next(
    (link for link in network_links if link["process_creation_found"]),
    None,
)

if matched_example is not None:
    print("\nEşleşen bir örnek:")
    print(json.dumps(matched_example, indent=2, ensure_ascii=False))

In [ ]:
unmatched_links = [
    link for link in network_links
    if not link["process_creation_found"]
]

unmatched_process_counts = Counter(
    link["process_guid"] for link in unmatched_links
)

unmatched_images = {
    link["process_guid"]: link["image"]
    for link in unmatched_links
}

print("Eşleşmeyen ağ olayı:", len(unmatched_links))
print("Eşleşmeyen farklı süreç:", len(unmatched_process_counts))

for process_guid, count in unmatched_process_counts.most_common():
    print(f"\nOlay sayısı: {count}")
    print("ProcessGuid:", process_guid)
    print("Program:", unmatched_images[process_guid])

In [ ]:
from datetime import datetime
from google.colab import files

context_path = DATA_DIR / "process-context.jsonl"

if not context_path.is_file():
    print("process-context.jsonl dosyasını seç.")
    files.upload()

context_events = load_events(context_path)
session_start = datetime.fromisoformat(
    manifest["window"]["started_at"].replace("Z", "+00:00")
)

for event in context_events:
    if event["event_id"] != 1:
        raise ValueError("Bağlam dosyasında süreç oluşturma olayı bekleniyordu.")

    if (
        event["experiment_id"] != manifest["experiment_id"]
        or event["host_id"] != manifest["host"]["host_id"]
    ):
        raise ValueError("Bağlam kaydı farklı bir oturuma veya bilgisayara ait.")

    context_time = datetime.fromisoformat(event["utc_time"])

    if context_time >= session_start:
        raise ValueError("Bağlam kaydı oturum başlangıcından önce değil.")

full_process_index = build_process_index(
    process_events + context_events
)

extended_network_links = link_events_to_processes(
    network_events,
    full_process_index,
)

extended_matched_count = sum(
    link["process_creation_found"]
    for link in extended_network_links
)

remaining_links = [
    link for link in extended_network_links
    if not link["process_creation_found"]
]

print("Eklenen bağlam kaydı:", len(context_events))
print("Genişletilmiş süreç indeksi:", len(full_process_index))
print("Toplam ağ olayı:", len(extended_network_links))
print("Eşleşen ağ olayı:", extended_matched_count)
print("Eşleşmeyen ağ olayı:", len(remaining_links))
print(
    "Sayı kontrolü:",
    extended_matched_count + len(remaining_links) == len(network_events),
)

for process_guid, count in Counter(
    link["process_guid"] for link in remaining_links
).items():
    print(f"Kalan süreç: {process_guid} — {count} olay")

In [ ]:
network_process_links = []

for link in extended_network_links:
    process_guid = link["process_guid"]
    process_event = full_process_index.get(process_guid)

    if process_guid in process_index:
        source = "session"
    elif process_event is not None:
        source = "pre_session"
    else:
        source = "unresolved"

    enriched_link = link.copy()
    enriched_link["process_creation_source"] = source
    enriched_link["process_creation_record_id"] = (
        process_event["record_id"]
        if process_event is not None
        else None
    )
    enriched_link["process_creation_utc_time"] = (
        process_event["utc_time"]
        if process_event is not None
        else None
    )

    network_process_links.append(enriched_link)

output_path = DATA_DIR / "network-process-links.jsonl"

with output_path.open("w", encoding="utf-8", newline="\n") as file:
    for link in network_process_links:
        file.write(json.dumps(link, ensure_ascii=False) + "\n")

source_counts = Counter(
    link["process_creation_source"]
    for link in network_process_links
)

print("Kaydedilen bağlantı kaydı:", len(network_process_links))

for source, count in sorted(source_counts.items()):
    print(f"{source}: {count}")

print("Dosya:", output_path)

In [ ]:
from google.colab import files

saved_links = load_events(output_path)

original_record_counts = Counter(
    event["record_id"] for event in network_events
)

saved_record_counts = Counter(
    link["record_id"] for link in saved_links
)

if original_record_counts != saved_record_counts:
    raise ValueError(
        "Kaydedilen bağlantılarda eksik, fazla veya tekrarlanan kayıt var."
    )

saved_source_counts = Counter(
    link["process_creation_source"] for link in saved_links
)

if saved_source_counts != {
    "session": 53,
    "pre_session": 139,
    "unresolved": 2,
}:
    raise ValueError("Kaydedilen süreç bağlamı dağılımı uyuşmuyor.")

print("Dosya doğrulandı:", len(saved_links), "kayıt")
files.download(str(output_path))

In [ ]:
def summarize_network_by_process(network_events: list[dict]) -> list[dict]:
    groups = {}

    for event in network_events:
        process_guid = event["data"]["ProcessGuid"]
        groups.setdefault(process_guid, []).append(event)

    features = []

    for process_guid, process_events in groups.items():
        destination_ips = {
            event["data"]["DestinationIp"]
            for event in process_events
        }

        destination_ports = {
            int(event["data"]["DestinationPort"])
            for event in process_events
        }

        protocols = Counter(
            event["data"]["Protocol"].lower()
            for event in process_events
        )

        features.append({
            "process_guid": process_guid,
            "image": process_events[0]["data"]["Image"],
            "network_event_count": len(process_events),
            "unique_destination_ip_count": len(destination_ips),
            "unique_destination_port_count": len(destination_ports),
            "tcp_event_count": protocols["tcp"],
            "udp_event_count": protocols["udp"],
        })

    return sorted(
        features,
        key=lambda feature: feature["network_event_count"],
        reverse=True,
    )


network_features = summarize_network_by_process(network_events)

print("Özetlenen süreç:", len(network_features))
print(
    "Toplam ağ olayı:",
    sum(feature["network_event_count"] for feature in network_features),
)

print("\nEn fazla ağ olayı üreten sürecin özellikleri:")
print(json.dumps(network_features[0], indent=2, ensure_ascii=False))

In [ ]:
from datetime import datetime

window_start = datetime.fromisoformat(
    manifest["window"]["started_at"].replace("Z", "+00:00")
)
window_end = datetime.fromisoformat(
    manifest["window"]["ended_at"].replace("Z", "+00:00")
)

window_duration_seconds = (window_end - window_start).total_seconds()

if window_duration_seconds <= 0:
    raise ValueError("Oturum süresi sıfırdan büyük olmalı.")

enriched_network_features = []

for feature in network_features:
    event_count = feature["network_event_count"]

    if event_count <= 0:
        raise ValueError("Ağ özeti en az bir olay içermeli.")

    enriched_feature = feature.copy()
    enriched_feature["tcp_event_ratio"] = (
        feature["tcp_event_count"] / event_count
    )
    enriched_feature["udp_event_ratio"] = (
        feature["udp_event_count"] / event_count
    )
    enriched_feature["network_events_per_window_second"] = (
        event_count / window_duration_seconds
    )

    enriched_network_features.append(enriched_feature)

print("Oturum süresi:", round(window_duration_seconds, 3), "saniye")
print("Özetlenen süreç:", len(enriched_network_features))
print("\nİlk sürecin genişletilmiş özellikleri:")
print(json.dumps(enriched_network_features[0], indent=2, ensure_ascii=False))

In [ ]:
feature_records = []

for feature in enriched_network_features:
    record = {
        "schema_version": "0.1.0",
        "experiment_id": manifest["experiment_id"],
        "host_id": manifest["host"]["host_id"],
        "window_started_at": window_start.isoformat(),
        "window_ended_at": window_end.isoformat(),
        "window_duration_seconds": window_duration_seconds,
        **feature,
    }
    feature_records.append(record)

total_network_events = sum(
    record["network_event_count"]
    for record in feature_records
)

if total_network_events != len(network_events):
    raise ValueError("Özetlerdeki olay toplamı kaynak kayıtlarla eşleşmiyor.")

feature_path = DATA_DIR / "network-process-features.jsonl"

with feature_path.open("w", encoding="utf-8", newline="\n") as file:
    for record in feature_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

saved_features = load_events(feature_path)

if saved_features != feature_records:
    raise ValueError("Kaydedilen özellikler bellekteki kayıtlarla eşleşmiyor.")

print("Kaydedilen süreç özeti:", len(saved_features))
print("Temsil edilen ağ olayı:", total_network_events)
print("Dosya doğrulandı:", saved_features == feature_records)
print("Dosya:", feature_path)

In [ ]:
from google.colab import files

files.download(str(feature_path))

In [ ]:
def summarize_dns_by_process(dns_events: list[dict]) -> list[dict]:
    groups = {}

    for event in dns_events:
        process_guid = event["data"]["ProcessGuid"]
        groups.setdefault(process_guid, []).append(event)

    features = []

    for process_guid, process_dns_events in groups.items():
        query_names = {
            event["data"]["QueryName"].casefold().rstrip(".")
            for event in process_dns_events
        }

        status_counts = Counter(
            int(event["data"]["QueryStatus"])
            for event in process_dns_events
        )

        event_count = len(process_dns_events)
        zero_status_count = status_counts[0]
        nonzero_status_count = event_count - zero_status_count

        features.append({
            "process_guid": process_guid,
            "image": process_dns_events[0]["data"]["Image"],
            "dns_event_count": event_count,
            "unique_query_name_count": len(query_names),
            "query_status_zero_count": zero_status_count,
            "query_status_nonzero_count": nonzero_status_count,
            "query_status_nonzero_ratio": (
                nonzero_status_count / event_count
            ),
            "query_status_counts": {
                str(status): count
                for status, count in sorted(status_counts.items())
            },
        })

    return sorted(
        features,
        key=lambda feature: feature["dns_event_count"],
        reverse=True,
    )


dns_features = summarize_dns_by_process(dns_events)

total_dns_events = sum(
    feature["dns_event_count"]
    for feature in dns_features
)

if total_dns_events != len(dns_events):
    raise ValueError("DNS özetleri kaynak olay sayısıyla eşleşmiyor.")

print("DNS olayı bulunan süreç:", len(dns_features))
print("Toplam DNS olayı:", total_dns_events)
print("Sayı kontrolü:", total_dns_events == len(dns_events))

if dns_features:
    print("\nEn fazla DNS olayı üreten sürecin özellikleri:")
    print(json.dumps(dns_features[0], indent=2, ensure_ascii=False))

In [ ]:
from collections import Counter
import json


def summarize_dns_by_process(dns_events: list[dict]) -> list[dict]:
    groups = {}

    for event in dns_events:
        process_guid = event["data"]["ProcessGuid"]
        groups.setdefault(process_guid, []).append(event)

    features = []

    for process_guid, process_events in groups.items():
        query_names = {
            event["data"]["QueryName"].strip().lower()
            for event in process_events
        }

        statuses = Counter(
            int(event["data"]["QueryStatus"])
            for event in process_events
        )

        event_count = len(process_events)
        zero_status_count = statuses[0]
        nonzero_status_count = event_count - zero_status_count

        features.append({
            "process_guid": process_guid,
            "image": process_events[0]["data"]["Image"],
            "dns_event_count": event_count,
            "unique_query_name_count": len(query_names),
            "dns_zero_status_count": zero_status_count,
            "dns_nonzero_status_count": nonzero_status_count,
            "dns_nonzero_status_ratio": (
                nonzero_status_count / event_count
            ),
            "query_status_counts": {
                str(status): count
                for status, count in sorted(statuses.items())
            },
        })

    return sorted(
        features,
        key=lambda feature: feature["dns_event_count"],
        reverse=True,
    )


dns_features = summarize_dns_by_process(dns_events)

total_dns_events = sum(
    feature["dns_event_count"]
    for feature in dns_features
)

if total_dns_events != len(dns_events):
    raise ValueError("DNS özetindeki olay toplamı kaynakla eşleşmiyor.")

print("Özetlenen DNS süreci:", len(dns_features))
print("Toplam DNS olayı:", total_dns_events)
print("Sayı kontrolü:", total_dns_events == len(dns_events))

if dns_features:
    print("\nEn fazla DNS olayı üreten sürecin özellikleri:")
    print(json.dumps(dns_features[0], indent=2, ensure_ascii=False))

In [ ]:
network_feature_index = {
    feature["process_guid"]: feature
    for feature in enriched_network_features
}

dns_feature_index = {
    feature["process_guid"]: feature
    for feature in dns_features
}

all_process_guids = (
    set(network_feature_index) | set(dns_feature_index)
)

combined_process_features = []

for process_guid in sorted(all_process_guids):
    network = network_feature_index.get(process_guid)
    dns = dns_feature_index.get(process_guid)

    record = {
        "process_guid": process_guid,
        "image": (
            network["image"]
            if network is not None
            else dns["image"]
        ),
        "has_network_events": network is not None,
        "has_dns_events": dns is not None,
        "network_event_count": 0,
        "unique_destination_ip_count": 0,
        "unique_destination_port_count": 0,
        "tcp_event_count": 0,
        "udp_event_count": 0,
        "tcp_event_ratio": None,
        "udp_event_ratio": None,
        "network_events_per_window_second": 0.0,
        "dns_event_count": 0,
        "unique_query_name_count": 0,
        "dns_zero_status_count": 0,
        "dns_nonzero_status_count": 0,
        "dns_nonzero_status_ratio": None,
        "query_status_counts": {},
    }

    if network is not None:
        record.update(network)

    if dns is not None:
        record.update(dns)

    combined_process_features.append(record)


network_total = sum(
    feature["network_event_count"]
    for feature in combined_process_features
)

dns_total = sum(
    feature["dns_event_count"]
    for feature in combined_process_features
)

if network_total != len(network_events):
    raise ValueError("Birleştirmede ağ olayı sayısı değişti.")

if dns_total != len(dns_events):
    raise ValueError("Birleştirmede DNS olayı sayısı değişti.")

both_count = sum(
    feature["has_network_events"] and feature["has_dns_events"]
    for feature in combined_process_features
)

network_only_count = sum(
    feature["has_network_events"] and not feature["has_dns_events"]
    for feature in combined_process_features
)

dns_only_count = sum(
    feature["has_dns_events"] and not feature["has_network_events"]
    for feature in combined_process_features
)

print("Birleştirilen süreç:", len(combined_process_features))
print("Ağ ve DNS olayı bulunan:", both_count)
print("Yalnızca ağ olayı bulunan:", network_only_count)
print("Yalnızca DNS olayı bulunan:", dns_only_count)
print("Toplam ağ olayı:", network_total)
print("Toplam DNS olayı:", dns_total)
print(
    "Süreç sayısı kontrolü:",
    both_count + network_only_count + dns_only_count
    == len(combined_process_features),
)

In [ ]:
contextual_process_features = []

for feature in combined_process_features:
    process_guid = feature["process_guid"]

    process_event = full_process_index.get(process_guid)

    if process_event is None:
        creation_source = "unresolved"
    elif process_guid in process_index:
        creation_source = "session"
    else:
        creation_source = "pre_session"

    process_data = (
        process_event["data"]
        if process_event is not None
        else {}
    )

    enriched_feature = feature.copy()

    enriched_feature.update({
        "process_creation_found": process_event is not None,
        "process_creation_source": creation_source,
        "process_creation_utc_time": (
            process_event["utc_time"]
            if process_event is not None
            else None
        ),
        "command_line": process_data.get("CommandLine"),
        "parent_process_guid": process_data.get("ParentProcessGuid"),
        "parent_image": process_data.get("ParentImage"),
    })

    contextual_process_features.append(enriched_feature)


source_counts = Counter(
    feature["process_creation_source"]
    for feature in contextual_process_features
)

if len(contextual_process_features) != len(combined_process_features):
    raise ValueError("Bağlam eklenirken süreç sayısı değişti.")

print("Bağlam eklenen süreç:", len(contextual_process_features))

for source in ("session", "pre_session", "unresolved"):
    print(f"{source}: {source_counts[source]}")

print(
    "Sayı kontrolü:",
    sum(source_counts.values()) == len(contextual_process_features),
)

print("\nOluşturma kaydı bulunamayan süreçler:")

for feature in contextual_process_features:
    if not feature["process_creation_found"]:
        print(feature["process_guid"], "—", feature["image"])

example = next(
    (
        feature
        for feature in contextual_process_features
        if feature["process_creation_found"]
    ),
    None,
)

if example is not None:
    print("\nBağlamı bulunan bir örnek:")
    print(json.dumps(example, indent=2, ensure_ascii=False))

In [ ]:
from datetime import datetime

additional_context_path = DATA_DIR / "process-context-additional.jsonl"

if not additional_context_path.is_file():
    raise FileNotFoundError(
        f"Ek bağlam dosyası bulunamadı: {additional_context_path}"
    )

additional_context_events = load_events(additional_context_path)

session_start = datetime.fromisoformat(
    manifest["window"]["started_at"].replace("Z", "+00:00")
)

for event in additional_context_events:
    if event["event_id"] != 1:
        raise ValueError("Ek bağlamda süreç oluşturma olayı bekleniyordu.")

    if (
        event["experiment_id"] != manifest["experiment_id"]
        or event["host_id"] != manifest["host"]["host_id"]
    ):
        raise ValueError("Ek bağlam kaydı farklı bir oturuma veya hosta ait.")

    event_time = datetime.fromisoformat(
        event["utc_time"].replace("Z", "+00:00")
    )

    if event_time.tzinfo is None:
        raise ValueError("Ek bağlam kaydının saat dilimi bilgisi eksik.")

    if event_time >= session_start:
        raise ValueError("Ek bağlam kaydı oturum başlangıcından önce değil.")


all_context_events = context_events + additional_context_events

updated_process_index = build_process_index(
    process_events + all_context_events
)

full_process_index = updated_process_index

active_process_guids = {
    feature["process_guid"]
    for feature in combined_process_features
}

matched_process_count = sum(
    process_guid in full_process_index
    for process_guid in active_process_guids
)

unmatched_process_guids = sorted(
    active_process_guids - set(full_process_index)
)

print("Önceki bağlam kaydı:", len(context_events))
print("Yeni bağlam kaydı:", len(additional_context_events))
print("Toplam bağlam kaydı:", len(all_context_events))
print("Güncellenen süreç indeksi:", len(full_process_index))
print("Ağ/DNS üreten süreç:", len(active_process_guids))
print("Oluşturma kaydı bulunan:", matched_process_count)
print("Oluşturma kaydı bulunamayan:", len(unmatched_process_guids))
print(
    "Sayı kontrolü:",
    matched_process_count + len(unmatched_process_guids)
    == len(active_process_guids),
)

for process_guid in unmatched_process_guids:
    print("Eşleşmeyen süreç:", process_guid)

In [ ]:
from collections import Counter
import json


def add_process_context(
    features: list[dict],
    full_index: dict[str, dict],
    session_index: dict[str, dict],
) -> list[dict]:
    contextual_features = []

    for feature in features:
        process_guid = feature["process_guid"]
        process_event = full_index.get(process_guid)

        if process_event is None:
            creation_source = "unresolved"
        elif process_guid in session_index:
            creation_source = "session"
        else:
            creation_source = "pre_session"

        process_data = (
            process_event["data"]
            if process_event is not None
            else {}
        )

        enriched_feature = feature.copy()

        enriched_feature.update({
            "process_creation_found": process_event is not None,
            "process_creation_source": creation_source,
            "process_creation_utc_time": (
                process_event["utc_time"]
                if process_event is not None
                else None
            ),
            "command_line": process_data.get("CommandLine"),
            "parent_process_guid": process_data.get("ParentProcessGuid"),
            "parent_image": process_data.get("ParentImage"),
        })

        contextual_features.append(enriched_feature)

    return contextual_features


contextual_process_features = add_process_context(
    combined_process_features,
    full_process_index,
    process_index,
)

source_counts = Counter(
    feature["process_creation_source"]
    for feature in contextual_process_features
)

network_total = sum(
    feature["network_event_count"]
    for feature in contextual_process_features
)

dns_total = sum(
    feature["dns_event_count"]
    for feature in contextual_process_features
)

if len(contextual_process_features) != len(combined_process_features):
    raise ValueError("Bağlam eklenirken süreç sayısı değişti.")

if network_total != len(network_events) or dns_total != len(dns_events):
    raise ValueError("Bağlam eklenirken olay toplamları değişti.")

print("Güncellenen süreç özeti:", len(contextual_process_features))

for source in ("session", "pre_session", "unresolved"):
    print(f"{source}: {source_counts[source]}")

print("Toplam ağ olayı:", network_total)
print("Toplam DNS olayı:", dns_total)

example = next(
    (
        feature
        for feature in contextual_process_features
        if feature["process_guid"]
        == "{14bdcb74-dc25-6aba-3200-000000000600}"
    ),
    None,
)

if example is not None:
    print("\nDaha önce eşleşmeyen DNS sürecinin güncel kaydı:")
    print(json.dumps(example, indent=2, ensure_ascii=False))

In [ ]:
import json

process_feature_records = []

for feature in contextual_process_features:
    record = {
        "schema_version": "0.2.0",
        "experiment_id": manifest["experiment_id"],
        "host_id": manifest["host"]["host_id"],
        "window_started_at": window_start.isoformat(),
        "window_ended_at": window_end.isoformat(),
        "window_duration_seconds": window_duration_seconds,
        **feature,
    }

    process_feature_records.append(record)


process_feature_path = DATA_DIR / "network-dns-process-features.jsonl"

with process_feature_path.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as file:
    for record in process_feature_records:
        file.write(
            json.dumps(record, ensure_ascii=False, allow_nan=False)
            + "\n"
        )


saved_process_features = load_events(process_feature_path)

if saved_process_features != process_feature_records:
    raise ValueError("Dosyadaki kayıtlar bellekteki kayıtlarla eşleşmiyor.")

saved_guids = [
    record["process_guid"]
    for record in saved_process_features
]

if len(saved_guids) != len(set(saved_guids)):
    raise ValueError("Dosyada tekrarlanan süreç GUID'si var.")

saved_network_total = sum(
    record["network_event_count"]
    for record in saved_process_features
)

saved_dns_total = sum(
    record["dns_event_count"]
    for record in saved_process_features
)

if saved_network_total != len(network_events):
    raise ValueError("Dosyadaki ağ olayı toplamı uyuşmuyor.")

if saved_dns_total != len(dns_events):
    raise ValueError("Dosyadaki DNS olayı toplamı uyuşmuyor.")

print("Kaydedilen süreç özeti:", len(saved_process_features))
print("Temsil edilen ağ olayı:", saved_network_total)
print("Temsil edilen DNS olayı:", saved_dns_total)
print("Dosya doğrulandı:", saved_process_features == process_feature_records)
print("Dosya:", process_feature_path)

In [ ]:
feature_index = {
    feature["process_guid"]: feature.copy()
    for feature in combined_process_features
}

added_process_count = 0

for event in process_events:
    process_guid = event["data"]["ProcessGuid"]

    if process_guid in feature_index:
        continue

    feature_index[process_guid] = {
        "process_guid": process_guid,
        "image": event["data"]["Image"],
        "has_network_events": False,
        "has_dns_events": False,
        "network_event_count": 0,
        "unique_destination_ip_count": 0,
        "unique_destination_port_count": 0,
        "tcp_event_count": 0,
        "udp_event_count": 0,
        "tcp_event_ratio": None,
        "udp_event_ratio": None,
        "network_events_per_window_second": 0.0,
        "dns_event_count": 0,
        "unique_query_name_count": 0,
        "dns_zero_status_count": 0,
        "dns_nonzero_status_count": 0,
        "dns_nonzero_status_ratio": None,
        "query_status_counts": {},
    }

    added_process_count += 1


all_process_features = add_process_context(
    list(feature_index.values()),
    full_process_index,
    process_index,
)

all_source_counts = Counter(
    feature["process_creation_source"]
    for feature in all_process_features
)

all_network_total = sum(
    feature["network_event_count"]
    for feature in all_process_features
)

all_dns_total = sum(
    feature["dns_event_count"]
    for feature in all_process_features
)

expected_guids = (
    set(process_index)
    | {
        feature["process_guid"]
        for feature in combined_process_features
    }
)

actual_guids = {
    feature["process_guid"]
    for feature in all_process_features
}

if actual_guids != expected_guids:
    raise ValueError("Genel özette eksik veya fazla süreç var.")

if all_network_total != len(network_events):
    raise ValueError("Genel özette ağ olayı toplamı değişti.")

if all_dns_total != len(dns_events):
    raise ValueError("Genel özette DNS olayı toplamı değişti.")

print("Önceki ağ/DNS süreç özeti:", len(combined_process_features))
print("Eklenen süreç:", added_process_count)
print("Genel süreç özeti:", len(all_process_features))

for source in ("session", "pre_session", "unresolved"):
    print(f"{source}: {all_source_counts[source]}")

print("Toplam ağ olayı:", all_network_total)
print("Toplam DNS olayı:", all_dns_total)
print("Süreç kapsamı kontrolü:", actual_guids == expected_guids)

whoami_feature = next(
    (
        feature
        for feature in all_process_features
        if feature["process_guid"]
        == "{14bdcb74-dd88-6aba-1c01-000000000600}"
    ),
    None,
)

if whoami_feature is not None:
    print("\nwhoami sürecinin genel özeti:")
    print(json.dumps(whoami_feature, indent=2, ensure_ascii=False))

In [ ]:
observed_child_counts = Counter(
    event["data"]["ParentProcessGuid"]
    for event in process_events
    if event["data"].get("ParentProcessGuid")
)

relational_process_features = []

for feature in all_process_features:
    process_guid = feature["process_guid"]
    parent_guid = feature["parent_process_guid"]

    enriched_feature = feature.copy()

    enriched_feature["observed_child_process_count"] = (
        observed_child_counts[process_guid]
    )

    enriched_feature["parent_process_creation_found"] = (
        parent_guid in full_process_index
        if parent_guid
        else None
    )

    relational_process_features.append(enriched_feature)


if len(relational_process_features) != len(all_process_features):
    raise ValueError("İlişki özellikleri eklenirken süreç sayısı değişti.")

print("İlişki özellikleri eklenen süreç:", len(relational_process_features))

print("\nwhoami ve onu başlatan cmd süreci:")

example_guids = {
    "{14bdcb74-dd88-6aba-1c01-000000000600}",
    "{14bdcb74-dd87-6aba-1b01-000000000600}",
}

for feature in relational_process_features:
    if feature["process_guid"] in example_guids:
        example = {
            "image": feature["image"],
            "command_line": feature["command_line"],
            "parent_image": feature["parent_image"],
            "observed_child_process_count": (
                feature["observed_child_process_count"]
            ),
            "parent_process_creation_found": (
                feature["parent_process_creation_found"]
            ),
        }

        print(json.dumps(example, indent=2, ensure_ascii=False))

In [ ]:
import json

general_process_records = []

for feature in relational_process_features:
    record = {
        "schema_version": "0.3.0",
        "experiment_id": manifest["experiment_id"],
        "host_id": manifest["host"]["host_id"],
        "window_started_at": window_start.isoformat(),
        "window_ended_at": window_end.isoformat(),
        "window_duration_seconds": window_duration_seconds,
        **feature,
    }

    general_process_records.append(record)


general_process_path = DATA_DIR / "process-features.jsonl"

with general_process_path.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as file:
    for record in general_process_records:
        file.write(
            json.dumps(record, ensure_ascii=False, allow_nan=False)
            + "\n"
        )


saved_general_features = load_events(general_process_path)

if saved_general_features != general_process_records:
    raise ValueError("Kaydedilen süreç özellikleri bellektekiyle eşleşmiyor.")

saved_guids = [
    record["process_guid"]
    for record in saved_general_features
]

if len(saved_guids) != len(set(saved_guids)):
    raise ValueError("Tekrarlanan süreç GUID'si bulundu.")

expected_guids = (
    set(process_index)
    | {
        feature["process_guid"]
        for feature in combined_process_features
    }
)

if set(saved_guids) != expected_guids:
    raise ValueError("Dosyada eksik veya fazla süreç var.")

network_total = sum(
    record["network_event_count"]
    for record in saved_general_features
)

dns_total = sum(
    record["dns_event_count"]
    for record in saved_general_features
)

if network_total != len(network_events):
    raise ValueError("Ağ olayı toplamı uyuşmuyor.")

if dns_total != len(dns_events):
    raise ValueError("DNS olayı toplamı uyuşmuyor.")

source_counts = Counter(
    record["process_creation_source"]
    for record in saved_general_features
)

print("Kaydedilen genel süreç özeti:", len(saved_general_features))
print("Temsil edilen ağ olayı:", network_total)
print("Temsil edilen DNS olayı:", dns_total)

for source in ("session", "pre_session", "unresolved"):
    print(f"{source}: {source_counts[source]}")

print("Dosya doğrulandı:", saved_general_features == general_process_records)
print("Dosya:", general_process_path)

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

bundle_names = [
    "manifest.json",
    "analysis-summary.json",
    "events.jsonl",
    "process-context.jsonl",
    "process-context-additional.jsonl",
    "network-dns-process-features.jsonl",
    "process-features.jsonl",
]

missing_files = [
    name
    for name in bundle_names
    if not (DATA_DIR / name).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Paket için eksik dosyalar: {missing_files}"
    )

bundle_path = DATA_DIR / (
    f"{manifest['experiment_id']}-analysis.zip"
)

with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in bundle_names:
        archive.write(
            DATA_DIR / name,
            arcname=name,
        )

with ZipFile(bundle_path, "r") as archive:
    damaged_file = archive.testzip()

    if damaged_file is not None:
        raise ValueError(f"ZIP içinde bozuk dosya: {damaged_file}")

    if sorted(archive.namelist()) != sorted(bundle_names):
        raise ValueError("ZIP dosyasının içeriği beklenen listeyle uyuşmuyor.")

print("Paketlenen dosya:", len(bundle_names))
print("Paket doğrulandı: True")
print("Dosya:", bundle_path)

files.download(str(bundle_path))

In [ ]:
from datetime import datetime
from collections import Counter
import json


def parse_utc_time(value: str) -> datetime:
    parsed_time = datetime.fromisoformat(
        value.replace("Z", "+00:00")
    )

    if parsed_time.tzinfo is None:
        raise ValueError("Zaman alanında saat dilimi bilgisi eksik.")

    return parsed_time


def validate_session_events(
    events: list[dict],
    manifest: dict,
) -> dict:
    start = parse_utc_time(manifest["window"]["started_at"])
    end = parse_utc_time(manifest["window"]["ended_at"])

    if end <= start:
        raise ValueError("Oturum bitişi başlangıcından sonra olmalı.")

    expected_experiment_id = manifest["experiment_id"]
    expected_host_id = manifest["host"]["host_id"]

    seen_record_ids = set()

    for row_number, event in enumerate(events, start=1):
        if event["experiment_id"] != expected_experiment_id:
            raise ValueError(
                f"{row_number}. kayıt farklı bir oturuma ait."
            )

        if event["host_id"] != expected_host_id:
            raise ValueError(
                f"{row_number}. kayıt farklı bir bilgisayara ait."
            )

        event_time = parse_utc_time(event["utc_time"])

        if not start <= event_time <= end:
            raise ValueError(
                f"{row_number}. kayıt oturum zaman aralığının dışında."
            )

        record_id = event["record_id"]

        if record_id in seen_record_ids:
            raise ValueError(
                f"Tekrarlanan kayıt kimliği: {record_id}"
            )

        seen_record_ids.add(record_id)

    event_counts = Counter(
        event["event_id"]
        for event in events
    )

    return {
        "experiment_id": expected_experiment_id,
        "host_id": expected_host_id,
        "window_duration_seconds": (end - start).total_seconds(),
        "event_count": len(events),
        "event_type_counts": {
            str(event_id): count
            for event_id, count in sorted(event_counts.items())
        },
        "validation_passed": True,
    }


session_validation = validate_session_events(events, manifest)

print(json.dumps(session_validation, indent=2, ensure_ascii=False))

In [ ]:
def build_session_process_features(
    events: list[dict],
    manifest: dict,
    context_events: list[dict],
) -> list[dict]:
    validation = validate_session_events(events, manifest)

    start = parse_utc_time(manifest["window"]["started_at"])
    end = parse_utc_time(manifest["window"]["ended_at"])
    duration = validation["window_duration_seconds"]

    for event in context_events:
        if event["event_id"] != 1:
            raise ValueError("Bağlamda süreç oluşturma olayı bekleniyordu.")

        if (
            event["experiment_id"] != manifest["experiment_id"]
            or event["host_id"] != manifest["host"]["host_id"]
        ):
            raise ValueError("Bağlam farklı bir oturuma veya hosta ait.")

        if parse_utc_time(event["utc_time"]) >= start:
            raise ValueError("Bağlam kaydı oturum başlangıcından önce değil.")

    session_processes = filter_events(events, 1)
    session_network = filter_events(events, 3)
    session_dns = filter_events(events, 22)

    session_index = build_process_index(session_processes)
    full_index = build_process_index(
        session_processes + context_events
    )

    network_summaries = summarize_network_by_process(session_network)
    dns_summaries = summarize_dns_by_process(session_dns)

    for feature in network_summaries:
        count = feature["network_event_count"]

        feature["tcp_event_ratio"] = feature["tcp_event_count"] / count
        feature["udp_event_ratio"] = feature["udp_event_count"] / count
        feature["network_events_per_window_second"] = count / duration

    network_index = {
        feature["process_guid"]: feature
        for feature in network_summaries
    }

    dns_index = {
        feature["process_guid"]: feature
        for feature in dns_summaries
    }

    active_guids = (
        set(session_index)
        | set(network_index)
        | set(dns_index)
    )

    features = []

    for guid in sorted(active_guids):
        network = network_index.get(guid)
        dns = dns_index.get(guid)

        image = (
            network["image"] if network is not None
            else dns["image"] if dns is not None
            else session_index[guid]["data"]["Image"]
        )

        feature = {
            "process_guid": guid,
            "image": image,
            "has_network_events": network is not None,
            "has_dns_events": dns is not None,
            "network_event_count": 0,
            "unique_destination_ip_count": 0,
            "unique_destination_port_count": 0,
            "tcp_event_count": 0,
            "udp_event_count": 0,
            "tcp_event_ratio": None,
            "udp_event_ratio": None,
            "network_events_per_window_second": 0.0,
            "dns_event_count": 0,
            "unique_query_name_count": 0,
            "dns_zero_status_count": 0,
            "dns_nonzero_status_count": 0,
            "dns_nonzero_status_ratio": None,
            "query_status_counts": {},
        }

        if network is not None:
            feature.update(network)

        if dns is not None:
            feature.update(dns)

        features.append(feature)

    features = add_process_context(
        features,
        full_index,
        session_index,
    )

    child_counts = Counter(
        event["data"]["ParentProcessGuid"]
        for event in session_processes
        if event["data"].get("ParentProcessGuid")
    )

    records = []

    for feature in features:
        parent_guid = feature["parent_process_guid"]

        record = {
            "schema_version": "0.3.0",
            "experiment_id": manifest["experiment_id"],
            "host_id": manifest["host"]["host_id"],
            "window_started_at": start.isoformat(),
            "window_ended_at": end.isoformat(),
            "window_duration_seconds": duration,
            **feature,
            "observed_child_process_count": (
                child_counts[feature["process_guid"]]
            ),
            "parent_process_creation_found": (
                parent_guid in full_index
                if parent_guid
                else None
            ),
        }

        records.append(record)

    if sum(r["network_event_count"] for r in records) != len(session_network):
        raise ValueError("Ağ olayı toplamı korunmadı.")

    if sum(r["dns_event_count"] for r in records) != len(session_dns):
        raise ValueError("DNS olayı toplamı korunmadı.")

    return records


session_process_records = build_session_process_features(
    events,
    manifest,
    all_context_events,
)

previous_by_guid = {
    record["process_guid"]: record
    for record in saved_general_features
}

new_by_guid = {
    record["process_guid"]: record
    for record in session_process_records
}

if new_by_guid != previous_by_guid:
    raise ValueError("Genel fonksiyonun çıktısı önceki analizle farklı.")

print("Üretilen süreç özeti:", len(session_process_records))
print("Önceki analizle aynı:", new_by_guid == previous_by_guid)

print(
    "Ağ olayı:",
    sum(r["network_event_count"] for r in session_process_records),
)

print(
    "DNS olayı:",
    sum(r["dns_event_count"] for r in session_process_records),
)

In [ ]:
from pathlib import Path
import json


def analyze_session_directory(
    session_dir: Path,
    output_dir: Path,
) -> dict:
    session_dir = Path(session_dir)
    output_dir = Path(output_dir)

    manifest_path = session_dir / "manifest.json"
    events_path = session_dir / "events.jsonl"

    for path in (manifest_path, events_path):
        if not path.is_file():
            raise FileNotFoundError(f"Gerekli dosya bulunamadı: {path}")

    session_manifest = json.loads(
        manifest_path.read_text(encoding="utf-8-sig")
    )

    session_events = load_events(events_path)

    context_names = [
        "process-context.jsonl",
        "process-context-additional.jsonl",
    ]

    session_context = []
    loaded_context_files = []

    for name in context_names:
        path = session_dir / name

        if path.is_file():
            session_context.extend(load_events(path))
            loaded_context_files.append(name)

    records = build_session_process_features(
        session_events,
        session_manifest,
        session_context,
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "process-features.jsonl"

    with output_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    allow_nan=False,
                )
                + "\n"
            )

    saved_records = load_events(output_path)

    if saved_records != records:
        raise ValueError("Kaydedilen sonuç analiz çıktısıyla eşleşmiyor.")

    source_counts = Counter(
        record["process_creation_source"]
        for record in saved_records
    )

    return {
        "experiment_id": session_manifest["experiment_id"],
        "session_event_count": len(session_events),
        "context_event_count": len(session_context),
        "loaded_context_files": loaded_context_files,
        "process_count": len(saved_records),
        "network_event_count": sum(
            record["network_event_count"]
            for record in saved_records
        ),
        "dns_event_count": sum(
            record["dns_event_count"]
            for record in saved_records
        ),
        "process_creation_sources": dict(source_counts),
        "output_verified": True,
        "output_path": str(output_path),
    }


analysis_result = analyze_session_directory(
    DATA_DIR,
    DATA_DIR / "analysis-results",
)

print(json.dumps(analysis_result, indent=2, ensure_ascii=False))

In [ ]:
without_context_records = build_session_process_features(
    events,
    manifest,
    [],
)

with_context_index = {
    record["process_guid"]: record
    for record in session_process_records
}

without_context_index = {
    record["process_guid"]: record
    for record in without_context_records
}

if set(with_context_index) != set(without_context_index):
    raise ValueError("Bağlam çıkarılınca süreç kapsamı değişti.")

observation_fields = [
    "network_event_count",
    "dns_event_count",
    "observed_child_process_count",
]

observations_preserved = all(
    with_context_index[guid][field]
    == without_context_index[guid][field]
    for guid in with_context_index
    for field in observation_fields
)

if not observations_preserved:
    raise ValueError("Bağlam çıkarılınca gözlenen olay sayıları değişti.")

with_context_counts = Counter(
    record["process_creation_source"]
    for record in session_process_records
)

without_context_counts = Counter(
    record["process_creation_source"]
    for record in without_context_records
)

print("Kategori       Bağlamlı   Bağlamsız")

for source in ("session", "pre_session", "unresolved"):
    print(
        f"{source:<14}"
        f"{with_context_counts[source]:>8}"
        f"{without_context_counts[source]:>11}"
    )

lost_context_count = sum(
    with_context_index[guid]["process_creation_found"]
    and not without_context_index[guid]["process_creation_found"]
    for guid in with_context_index
)

print("\nSüreç kapsamı aynı:", True)
print("Gözlenen olay sayıları aynı:", observations_preserved)
print("Oluşturma bağlamını kaybeden süreç:", lost_context_count)

In [ ]:
import pandas as pd

feature_columns = [
    "network_event_count",
    "unique_destination_ip_count",
    "unique_destination_port_count",
    "tcp_event_count",
    "udp_event_count",
    "tcp_event_ratio",
    "udp_event_ratio",
    "network_events_per_window_second",
    "dns_event_count",
    "unique_query_name_count",
    "dns_zero_status_count",
    "dns_nonzero_status_count",
    "dns_nonzero_status_ratio",
    "observed_child_process_count",
]

metadata_columns = [
    "experiment_id",
    "host_id",
    "process_guid",
    "image",
    "command_line",
    "parent_process_guid",
    "parent_image",
    "process_creation_source",
    "parent_process_creation_found",
]

feature_rows = []
metadata_rows = []

for record in session_process_records:
    feature_rows.append({
        column: record[column]
        for column in feature_columns
    })

    metadata_rows.append({
        column: record[column]
        for column in metadata_columns
    })

model_feature_table = pd.DataFrame(
    feature_rows,
    columns=feature_columns,
)

process_metadata_table = pd.DataFrame(
    metadata_rows,
    columns=metadata_columns,
)

if len(model_feature_table) != len(process_metadata_table):
    raise ValueError("Özellik ve metadata satır sayıları farklı.")

print("Özellik tablosunun boyutu:", model_feature_table.shape)
print("Metadata satırı:", len(process_metadata_table))

print("\nEksik değer bulunan özellikler:")

missing_counts = model_feature_table.isna().sum()

print(missing_counts[missing_counts > 0].to_string())

print("\nİlk üç sürecin sayısal özellikleri:")
print(model_feature_table.head(3).to_string(index=False))